# Comparativa práctica de modelos 2025

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/benchmarks/03-comparativa-modelos.ipynb)

Benchmark automatizado multi-proveedor: velocidad, calidad y coste.

In [ ]:
!pip install anthropic openai matplotlib -q

In [ ]:
import anthropic
import time
import matplotlib.pyplot as plt
import pandas as pd

client = anthropic.Anthropic()

PROMPTS_TEST = [
    'Explica el gradiente descendente en 2 frases.',
    'Escribe una función Python para invertir una cadena.',
    '¿Cuánto es 127 × 43? Muestra los pasos.',
    'Clasifica como positivo/negativo: "La comida estaba fría y el servicio fue lento."',
    'Resume en 1 frase: "El cambio climático afecta la biodiversidad global."',
]

## 1. Benchmark de velocidad (modelos Anthropic)

In [ ]:
MODELOS_ANTHROPIC = {
    'Haiku 4.5': 'claude-haiku-4-5-20251001',
    'Sonnet 4.6': 'claude-sonnet-4-6',
}

resultados = {}
for nombre, modelo_id in MODELOS_ANTHROPIC.items():
    latencias = []
    for prompt in PROMPTS_TEST:
        inicio = time.time()
        client.messages.create(
            model=modelo_id,
            max_tokens=128,
            messages=[{'role': 'user', 'content': prompt}],
        )
        latencias.append(time.time() - inicio)
    resultados[nombre] = {
        'latencia_media': sum(latencias) / len(latencias),
        'latencia_min': min(latencias),
        'latencia_max': max(latencias),
    }
    print(f"{nombre}: media={resultados[nombre]['latencia_media']:.2f}s, min={resultados[nombre]['latencia_min']:.2f}s")

## 2. Visualización comparativa

In [ ]:
# Datos de benchmarks públicos y precios (estáticos para la demo)
datos_comparativa = pd.DataFrame({
    'Modelo': ['Claude Opus 4.7', 'Claude Sonnet 4.6', 'Claude Haiku 4.5', 'GPT-4o', 'Gemini 1.5 Pro'],
    'MMLU': [90, 85, 78, 88, 86],
    'Precio Input $/M': [15.0, 3.0, 0.80, 2.50, 3.50],
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: MMLU scores
colores = ['#1565C0', '#1976D2', '#42A5F5', '#4CAF50', '#FF9800']
ax1.barh(datos_comparativa['Modelo'], datos_comparativa['MMLU'], color=colores)
ax1.set_xlabel('MMLU Score (%)')
ax1.set_title('Puntuación MMLU por modelo')
ax1.set_xlim(70, 95)
for i, v in enumerate(datos_comparativa['MMLU']):
    ax1.text(v + 0.3, i, f'{v}%', va='center', fontsize=9)

# Gráfico 2: Precio vs calidad
ax2.scatter(datos_comparativa['Precio Input $/M'], datos_comparativa['MMLU'],
           s=200, c=colores, zorder=5)
for _, row in datos_comparativa.iterrows():
    ax2.annotate(row['Modelo'].replace('Claude ', '').replace('Gemini ', 'Gem. '),
                (row['Precio Input $/M'], row['MMLU']),
                textcoords='offset points', xytext=(5, 5), fontsize=7)
ax2.set_xlabel('Precio Input ($/millón tokens)')
ax2.set_ylabel('MMLU Score (%)')
ax2.set_title('Calidad vs Precio')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()